### I. Downloading the Dataset

In [ ]:
from pathlib import Path
import pandas as pd

# metadata configs
split_mode = 'closed'
segment = 'head'

DATASET_DIR = './data/turtle-data'
METADATA_FILE = Path(DATASET_DIR) / f'metadata_splits_{segment}.csv'
metadata_df = pd.read_csv(METADATA_FILE)
metadata_df['file_name'] = metadata_df['file_name'].apply(lambda x: str(Path(DATASET_DIR) / x))
metadata_df = metadata_df[['file_name', 'identity', f'split_{split_mode}', 'bounding_box']]
metadata_df = metadata_df.rename(columns={'file': 'file_name', 'id': 'identity'})
identities = metadata_df["identity"].unique().tolist()
label_to_index = {label: idx for idx, label in enumerate(identities)}
identities = {label_to_index[label]: label for label in identities}
metadata_df['label'] = metadata_df['identity'].map(label_to_index)
print(metadata_df.head(-10))

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import ast
import torchvision.transforms as T

class SeaTurtleDataset(Dataset):
    def __init__(self, df, split_mode, split, transform=None):
        self.df = df[df[f'split_{split_mode}'] == split].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "file_name"]
        label = int(self.df.loc[idx, "label"])
        img = Image.open(img_path).convert("RGB")
        bbox = self.df.loc[idx, "bounding_box"]
        if isinstance(bbox, str):
            bbox = ast.literal_eval(bbox)
        if bbox is not None:
            x, y, w, h = bbox
            x1, y1 = int(x), int(y)
            x2, y2 = int(x + w), int(y + h)
            img = img.crop((x1, y1, x2, y2))
        if self.transform:
            img = self.transform(img)
        return img, label
    
transforms_train = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transforms_test  = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
    
train_set = SeaTurtleDataset(metadata_df, split_mode, 'train', transform=transforms_train)
val_set = SeaTurtleDataset(metadata_df, split_mode, 'valid', transform=transforms_test)
test_set = SeaTurtleDataset(metadata_df, split_mode, 'test', transform=transforms_test)
print("Train/Val/Test sizes:", len(train_set), len(val_set), len(test_set))

In [ ]:
import matplotlib.pyplot as plt
import random

def display_sample_images(dataset, num_images=5):
    plt.figure(figsize=(15, 5))
    for i in range(num_images):
        random_index = random.randint(0, len(dataset) - 1)
        image, label = dataset[random_index]
        image = image.permute(1, 2, 0).numpy()
        image = image * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # unnormalize
        image = image.clip(0, 1)
        plt.subplot(1, num_images, i + 1)
        plt.imshow(image)
        plt.title(f'Label: {identities[label]}')
        plt.axis('off')
    plt.show()

display_sample_images(train_set, num_images=5)

### II. Building the Model

In [ ]:
from torchvision.models import convnext_base, ConvNeXt_Base_Weights
import torch.nn as nn
import torch
from sklearn.preprocessing import normalize
import math

class ConvNeXtBackbone(nn.Module):
    def __init__(self, embedding_dim=512, pretrained=True, dropout=0.1):
        super().__init__()
        weights = ConvNeXt_Base_Weights.IMAGENET1K_V1 if pretrained else None
        model = convnext_base(weights=weights)
        
        in_dim = model.classifier[2].in_features
        model.classifier[2] = nn.Identity()
        self.backbone = model
        
        # Projection with dropout
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(in_dim, embedding_dim)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.dropout(feat)
        emb = self.proj(feat)
        return emb
    
class ArcFace(nn.Module):
    def __init__(self, embedding_size, classnum, s=64.0, m=0.5):
        super().__init__()
        self.classnum = classnum
        self.kernel = nn.Parameter(torch.Tensor(embedding_size, classnum))
        self.kernel.data.uniform_(-1, 1).renorm_(2, 1, 1e-5).mul_(1e5)
        self.m = m
        self.s = s
        self.eps = 1e-4

    def forward(self, embeddings, label):
        kernel_norm = normalize(self.kernel, axis=0)
        cosine = torch.mm(embeddings, kernel_norm)
        cosine = cosine.clamp(-1+self.eps, 1-self.eps)

        m_hot = torch.zeros_like(label.size()[0], cosine.size()[1], device=cosine.device)
        m_hot.scatter_(1, label.view(-1, 1), self.m)

        theta = torch.acos(cosine)

        theta_m = torch.clip(theta + m_hot, min=self.eps, max=math.pi - self.eps)
        cosine_m = torch.cos(theta_m)
        scaled_cosine_m = cosine_m * self.s
        return scaled_cosine_m, embeddings

class ConvNeXtArcFaceModel(nn.Module):
    def __init__(self, num_classes, embedding_dim=512, dropout=0.1):
        super().__init__()
        self.backbone = ConvNeXtBackbone(
            embedding_dim=embedding_dim, 
            pretrained=True,
            dropout=dropout
        )
        self.head = ArcFace(
            embedding_size=embedding_dim,
            classnum=num_classes,
            s=64.0,
            m=0.5
        )

    def forward(self, x, labels=None):
        emb = self.backbone(x)                    # (B, D) - raw embeddings
        if labels is not None:
            logits, emb_norm = self.head(emb, labels)  # (B, C), (B, D)
            return logits, emb_norm
        else:
            logits, emb_norm = self.head(emb, None)
            return emb_norm  # Return normalized embeddings for inference

### III. Training Loop

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader

epochs = 20
learning_rate = 0.001
model = ConvNeXtArcFaceModel(num_classes=len(identities), embedding_dim=512, dropout=0.1)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)

BATCH_SIZE = 32
NUM_WORKERS = 4
PIN_MEMORY = True   
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

In [ ]:
import torch
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model.to(device)

early_stopping_patience = 5
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Training"):
        print(f"Processing batch with images shape: {images.shape} and labels shape: {labels.shape}")
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(images, labels)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {epoch_loss:.4f}")
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - Validation"):
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images, labels)
            loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
    val_loss /= len(val_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs}, Validation Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print("Model saved!")
    else:
        patience_counter += 1
        if patience_counter >= early_stopping_patience:
            print("Early stopping triggered.")
            break

### IV. Testing

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        logits, _ = model(images, labels)
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f"Test Accuracy: {100 * correct / total:.2f}%")

### V. Visualizing using t-sne

In [ ]:
from sklearn.manifold import TSNE
import numpy as np

@torch.no_grad()
def extract_embeddings(model, loader, device, max_points=2000):
    model.eval()
    all_emb = []
    all_lab = []

    count = 0
    for images, labels in tqdm(loader, desc="Embed"):
        images = images.to(device, non_blocking=True)
        _, emb = model(images, labels=None)   # inference mode
        all_emb.append(emb.cpu().numpy())
        all_lab.append(labels.numpy())
        count += labels.size(0)
        if count >= max_points:
            break

    emb = np.concatenate(all_emb, axis=0)[:max_points]
    lab = np.concatenate(all_lab, axis=0)[:max_points]
    return emb, lab

emb, lab = extract_embeddings(model, test_loader, device, max_points=2000)
print("Embeddings:", emb.shape, "Labels:", lab.shape)

tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=42)
emb_2d = tsne.fit_transform(emb)

plt.figure(figsize=(10, 8))
plt.scatter(emb_2d[:, 0], emb_2d[:, 1], s=8, c=lab, alpha=0.7)
plt.title("t-SNE of ConvNeXt + AdaFace Embeddings (Test subset)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Class label (index)")
plt.show()